# ModernBERT domain classifier (v4)

Fixes vs v3:
1. `compute_metrics` no longer shadows `labels=` into `f1_score`; removed meaningless `pos_label=1` for multi-class.
2. `push_to_hub=False` by default (avoids auth errors when the HF login cell is not run). Re-enable after logging in.
3. Dynamic padding via `DataCollatorWithPadding` instead of padding inside `dataset.map`.

**Security note:** the v3 notebook had a real HF token pasted into a commented-out cell. Even commented, it is committed to history — **revoke that token on huggingface.co and generate a new one** before pushing.

In [ ]:
# Install Hugging Face libraries
%pip install --upgrade \
  "datasets==3.1.0" \
  "accelerate==1.2.1" \
  "hf-transfer==0.1.8"

# ModernBERT is not yet available in an official release, so we need to install it from github
%pip install "git+https://github.com/huggingface/transformers.git@6e0515e99c39444caae39472ee1b2fd76ece32f1" --upgrade

In [ ]:
# Uncomment and paste a fresh token if you want to push to the Hub.
# Do NOT commit a real token — generate one at https://huggingface.co/settings/tokens
# from huggingface_hub import login
# login(token="hf_xxx_REPLACE_ME", add_to_git_credential=True)

In [ ]:
from datasets import load_dataset

dataset_id = "argilla/synthetic-domain-text-classification"

train_dataset = load_dataset(dataset_id, split="train")

split_dataset = train_dataset.train_test_split(test_size=0.1)
split_dataset["train"][0]

In [ ]:
from transformers import AutoTokenizer, DataCollatorWithPadding

model_id = "answerdotai/ModernBERT-base"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Tokenize without padding here; pad per-batch via DataCollatorWithPadding at training time.
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)

if "label" in split_dataset["train"].features.keys():
    split_dataset = split_dataset.rename_column("label", "labels")  # match Trainer
tokenized_dataset = split_dataset.map(tokenize, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenized_dataset["train"].features.keys()

In [ ]:
from transformers import AutoModelForSequenceClassification

model_id = "answerdotai/ModernBERT-base"

labels = tokenized_dataset["train"].features["labels"].names
num_labels = len(labels)
label2id = {label: str(i) for i, label in enumerate(labels)}
id2label = {str(i): label for i, label in enumerate(labels)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_id, num_labels=num_labels, label2id=label2id, id2label=id2label,
)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# Fixed: no more `labels=labels` shadowing, no meaningless `pos_label=1`.
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    score = f1_score(labels, predictions, average="weighted")
    return {"f1": float(score)}

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="ModernBERT-domain-classifier",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-5,
    num_train_epochs=5,
    bf16=True,
    optim="adamw_torch_fused",
    logging_strategy="steps",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    # Set push_to_hub=True only after running the login cell with a valid token.
    push_to_hub=False,
    hub_strategy="every_save",
    gradient_checkpointing=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()
tokenizer.save_pretrained("ModernBERT-domain-classifier")
trainer.create_model_card()
# Only call push_to_hub() after enabling push_to_hub=True above and logging in.
# trainer.push_to_hub()

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

model_id = "argilla/ModernBERT-domain-classifier"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(
    model_id, torch_dtype=torch.float16
).to("cuda")

classifier = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0,
)

classifier("Smoking is bad for your health.")

In [ ]:
samples = [
    "Smoking is bad for your health.",
    "Google is one of the largest tech companies.",
    "New agricultural tools have boosted rice production.",
]
classifier(samples, batch_size=8)